<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/01_agente_simple.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap01) · 🧭 [Teoría del capítulo](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap01-modelo-e-instrucciones.html) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 0](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/00_configuracion_cuenta_aws.ipynb) · ➡️ [Cap. 2](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/02_herramientas.ipynb)

# Capítulo 1 — El agente que sabe… lo que no es

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Laura escribe: *«¿Cuántos días tengo para devolver un producto?»*. El asistente responde enseguida y con seguridad… con el plazo legal general, no con el de TiendaNova.

| | |
|---|---|
| **🧠 Componente agéntico** | Modelo e instrucciones: el razonamiento |
| **La necesidad** | Un agente que hable bien, que se presente como IA y que no confunda lo que sabe en general con lo que es cierto en TiendaNova |
| **Lo que construimos** | Agente Strands con *system prompt*, modelo vía inference profile `eu.` y reintentos adaptativos |
| **🏠 En local** | ✅ Funciona: el agente corre en tu notebook (el modelo, eso sí, ya es un servicio remoto). |
| **☁️ Por qué AWS al salir a Internet** | En local usarías una clave de API en un `.env`: con muchas llamadas llegan los 429 (*throttling*) y no controlas dónde se procesan los datos. Bedrock usa IAM sin claves, el inference profile `eu.` reparte la carga entre regiones europeas y los reintentos adaptativos absorben picos. No elimina las cuotas: la aplicación debe saber degradarse. |
| **En el ordenador sería…** | La **ROM**: el modelo trae de fábrica mucho conocimiento general, pero es de solo lectura y no sabe nada de TiendaNova |
| **Requisitos del caso** | RNF-03 (identificarse como IA) |
| **Prerrequisito · duración** | Capítulo 0 · 40 min |
| **⚠️ Dónde se rompe la analogía** | Una ROM devuelve exactamente lo que guarda; el modelo **reconstruye** su conocimiento de forma probabilística. Cuando una pregunta se parece a algo que sabe, responde con lo general como si fuera lo particular. |

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=1)

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(1)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(1)
# ponerse_al_dia(1)   # ← descomenta si llegas tarde o algo se rompió

## 1. Experimento de rotura: un modelo sin datos de la empresa

El modelo es una ROM enorme: sabe redactar, razonar y mucho sobre el mundo, incluida la normativa europea de consumo. Lo que no puede saber es cómo funciona **TiendaNova**: su política de devoluciones da **30 días**, el doble del mínimo legal.

> ✋ **Predice antes de ejecutar.** Laura pregunta cuántos días tiene para devolver un producto. ¿Qué hará el modelo: dirá que no lo sabe, preguntará, o dará una cifra? Si da una cifra, ¿cuál? Apúntala.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

agente_sin_datos = Agent(
    model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0.2),
    system_prompt="Eres el asistente de atención al cliente de TiendaNova, una tienda online española.",
    callback_handler=None,
)
r = agente_sin_datos("Hola, ¿cuántos días tengo para devolver un producto?")
print("🤖", r)
medir(r, "sin-datos")

**Comprobación.** La respuesta del modelo cambia entre ejecuciones; esta celda mira qué cifra ha dado y la compara con la política real de TiendaNova (`datos/tiendanova.json`).

In [ ]:
import json, re

politica = next(d for d in json.loads((DATOS_DIR / "tiendanova.json").read_text(encoding="utf-8"))["documentos"]
                if d["doc_id"] == "politica_devoluciones")
print("Política de TiendaNova →", politica["contenido_md"].split("## Plazo")[1].split("##")[0].strip(), "\n")

cifras = set(re.findall(r"\b(\d{1,3})\s*d[ií]as", str(r)))
if "30" in cifras and "14" not in cifras:
    print("Ha dicho 30 días: coincide, pero no lo ha leído en ningún sitio. Repite la celda anterior: ¿lo mantiene?")
elif "14" in cifras:
    print("Ha respondido con el mínimo legal europeo (14 días): es lo que trae de fábrica. Para TiendaNova es falso: son 30.")
elif cifras:
    print(f"Ha dado {sorted(cifras)} días: una cifra plausible, ninguna sale de la política de TiendaNova.")
else:
    print("No ha dado cifra (ha preguntado o ha remitido a la web). Bien por la prudencia… pero sigue sin poder contestar.")

Lo habitual con los modelos actuales es la segunda rama: **14 días**, con seguridad y bien explicado. No es un dato inventado al azar: es conocimiento general correcto (el derecho de desistimiento en la UE) aplicado a un caso donde **no vale**. Es la forma más peligrosa de alucinación, porque suena verificada.

Compáralo con lo que pasa cuando la pregunta no se parece a nada que el modelo sepa:

In [ ]:
r = agente_sin_datos("¿Y dónde está mi pedido TN-1001?")
print("🤖", r)
medir(r, "sin-datos")

Sobre el pedido, lo normal es que diga que no tiene acceso: sabe que no puede saberlo. Sobre el plazo, cree que lo sabe. **El riesgo no es el modelo que dice «no lo sé», sino el que sabe algo parecido.**

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**La ROM del agente**](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap01-modelo-e-instrucciones.html). Hazle al modelo preguntas que están en su ROM, preguntas que no pueden estar y preguntas que se *parecen* a algo que sabe, con y sin reglas. Vuelve cuando sepas responder a esta pregunta: **¿qué sabe el modelo de fábrica, qué no puede saber de ninguna manera, y por qué es más peligrosa la pregunta que se parece a algo que sabe?**

**En dos líneas:** un agente es un sistema que persigue un objetivo en un bucle (razona, actúa, observa y vuelve a razonar), y **no es un modelo**: es un modelo rodeado de componentes que este curso irá encendiendo uno a uno. El modelo aporta razonamiento y un conocimiento general fijado en el entrenamiento (la ROM); las instrucciones fijan el rol y los límites; sin los datos de la empresa, completa con lo plausible, y lo plausible a veces es la regla general en lugar de la tuya. El mapa completo de componentes está en [teoria/index.html](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html).

## 2. Un agente con reglas (y preparado para producción)

Las reglas no añaden conocimiento: fijan el rol, el tono y los límites. Veamos si bastan para el plazo.

> ✋ **Predice antes de ejecutar.** Con la regla «si no dispones de un dato de TiendaNova, dilo», ¿qué responderá ahora a la pregunta del plazo?

In [ ]:
from botocore.config import Config

# Hilo de producción: reintentos adaptativos ante 429 (throttling) y errores transitorios
reintentos = Config(retries={"mode": "adaptive", "max_attempts": 8})

SYSTEM_PROMPT = """Eres Nova Assist, el asistente de postventa de TiendaNova.
- En tu primer mensaje te presentas como asistente de inteligencia artificial.
- Respondes en el idioma del cliente, con tono cercano y profesional.
- Si no dispones de un dato concreto de TiendaNova (plazos, estados de pedido, fechas, importes), lo dices
  claramente: nunca lo sustituyas por una regla general ni lo inventes."""

agente = Agent(
    model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0.2,
                       boto_client_config=reintentos),
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)
r = agente("Hola, ¿cuántos días tengo para devolver un producto?")
print("🤖", r)
medir(r, "con-reglas")

In [ ]:
texto = str(r).lower()
print("¿Se presenta como IA?            ", any(p in texto for p in ("inteligencia artificial", "asistente virtual", " ia ", "ia,")))
print("¿Da los 30 días de TiendaNova?   ", "30 días" in texto)
print("¿Reconoce que no tiene el dato?  ", any(p in texto for p in ("no dispongo", "no tengo", "no puedo confirmar",
                                                                   "consultar", "verificar")))

Con reglas, lo normal es que se presente como IA (art. 50 del AI Act) y que matice: quizá mencione el mínimo legal, pero avisando de que no conoce la política concreta. **Mejor, pero sigue sin poder decir «30 días».** Las reglas cambian *cómo* responde; no le dan *datos*. Anotamos la deuda: la pagaremos en el capítulo 8, cuando el agente pueda leer las políticas de TiendaNova y citarlas.

## 3. Hilo de producción: ¿qué pasa con muchas llamadas a la vez?

En producción, cientos de clientes escriben a la vez. Cada cuenta y región tiene **cuotas** de inferencia; si se superan, Bedrock responde con `ThrottlingException` (429). Para responder a eso hay varias capas:

1. **Inference profile `eu.`:** reparte la inferencia entre varias regiones europeas, lo que da más capacidad y resiliencia, sin sacar los datos de la UE.
2. **Reintentos adaptativos** con *backoff*: absorben los picos breves.
3. **Cuotas dimensionadas** (y ampliadas si hace falta) y, si se necesita capacidad garantizada, *provisioned throughput*.
4. **Degradación elegante:** si aun así falla, la aplicación responde algo útil en lugar de caerse.

Lanzamos 10 llamadas simultáneas. Con cuotas normales **probablemente no verás ningún 429**: el objetivo es ver el patrón, no tumbar la cuenta.

> ✋ **Predice antes de ejecutar.** Diez llamadas simultáneas al modelo: ¿cuántas fallarán con un 429? ¿Ninguna, una, la mitad? Apunta tu cifra.

In [ ]:
import time
import boto3
from concurrent.futures import ThreadPoolExecutor

rt = boto3.client("bedrock-runtime", region_name=region(), config=reintentos)

def llamada(i):
    t0 = time.perf_counter()
    try:
        rt.converse(modelId=state_get("MODEL_ID"),
                    messages=[{"role": "user", "content": [{"text": f"Responde solo con el número {i}"}]}],
                    inferenceConfig={"maxTokens": 5})
        return "ok", time.perf_counter() - t0
    except rt.exceptions.ThrottlingException:
        return "429", time.perf_counter() - t0

with ThreadPoolExecutor(max_workers=10) as ex:
    resultados = list(ex.map(llamada, range(10)))
print({k: sum(1 for r in resultados if r[0] == k) for k in ("ok", "429")},
      "· latencia máxima:", round(max(r[1] for r in resultados), 2), "s")

In [ ]:
def responder_con_degradacion(pregunta: str) -> str:
    """Patrón de degradación elegante: si la inferencia falla tras los reintentos, no se cae la aplicación."""
    try:
        return str(agente(pregunta))
    except Exception:  # noqa: BLE001  (en producción: registrar la traza y encolar)
        return "Ahora mismo tenemos mucha demanda. Te respondemos en unos minutos por tu canal preferido."

print(responder_con_degradacion("¿Qué horario tenéis?"))

## 4. Arranca el taxímetro

Cada respuesta muestra una línea 🚕 con los *tokens* consumidos. Si fijaste los precios del modelo en el capítulo 0 (`precios(...)`), también verás el coste estimado. Iremos comparando capítulo a capítulo.

In [ ]:
taximetro()

## Para pensar

- Las reglas del *system prompt* reducen el riesgo, pero no aportan datos. ¿Qué pieza falta?
- ¿Dónde debería vivir el requisito del art. 50 (identificarse como IA) para que no dependa de que alguien lo recuerde en cada cambio de prompt?

## 🔎 Lo que acabamos de descubrir

El agente ya es prudente… pero no **sabe** nada de TiendaNova: ni los plazos de sus políticas ni dónde está un pedido. Una ROM no basta; necesita **acceso a datos reales**. Empezamos el cuaderno de deudas con la primera.

In [ ]:
deuda("D0", "abierta", 1, "sin las políticas de TiendaNova: el plazo de 30 días no puede salir de la ROM")
cuaderno()
taximetro_cerrar(1)
guardar_partida(1)

📚 **Para repasar más adelante:** la teoría de este capítulo está en [`teoria/cap01-modelo-e-instrucciones.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/cap01-modelo-e-instrucciones.html) y el mapa completo en [`teoria/index.html`](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html). La referencia por componente es la Parte I del documento.

---
➡️ **Siguiente capítulo:** `02_herramientas.ipynb` — *El agente que no ve tus datos*

---
**Siguiente:** ➡️ [Capítulo 2 · El agente que no ve tus datos](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/02_herramientas.ipynb) · 🎬 [su escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap02). En Colab, tu estado ya está en Drive: el capítulo siguiente lo recupera solo.